In [1]:
import math
import requests
import folium
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(
    { "figure.figsize": (17, 7) },
    style='ticks',
    palette=sns.color_palette("Set2"),
    color_codes=True,
    font_scale=5
)

plt.rcParams.update({
    "axes.labelsize": 12,  # Axes label font size
})

%config InlineBackend.figure_format = 'retina'
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data
stops_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stops.txt")
stop_times_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stop_times.txt")
trips = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\trips.txt")

# Convert datatypes
stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

# Merge route id details into stop times
stop_times_df = stop_times_df.merge(
    trips[["route_id", "trip_id"]],
    left_on='trip_id', 
    right_on='trip_id', 
    how='left')

# Isolate Bradford
# stops_df = stops_df.loc[(stops_df["stop_lat"] < 54) & (stops_df["stop_lat"] > 53.7)].reset_index(drop=True)
# stops_df = stops_df.loc[(stops_df["stop_lon"] < -1.55) & (stops_df["stop_lon"] > -1.95)].reset_index(drop=True)

# Get routes through each stop

In [3]:
stop_route_dict = stop_times_df.groupby("stop_id")["route_id"].unique().to_dict()
route_stops_dict = stop_times_df.groupby("route_id")["stop_id"].unique().to_dict()

# Find route v2

In [4]:
origin = (53.79, -1.79)
dest = (53.841401, -1.827540)

In [5]:
lon_scale = math.cos(math.radians(origin[0]))

## Route Timing

In [6]:
# docker run -t -i -p 5001:5000 -v "${PWD}:/data" osrm/osrm-backend osrm-routed --algorithm mld /data/west-yorkshire-foot.osrm

def time_journey(origin, dest):
    o_str = f"{origin[1]},{origin[0]}"
    d_str = f"{dest[1]},{dest[0]}"
    
    url = f"http://127.0.0.1:5001/route/v1/foot/{o_str};{d_str}?overview=false"
    
    response = requests.get(url).json()
    return response['routes'][0]['duration']

In [7]:
stops_df['rough_dist_origin'] = (stops_df["stop_lat"] - origin[0])**2 + ((stops_df["stop_lon"] - origin[1]) * lon_scale)**2
stops_df['rough_dist_dest'] = (stops_df["stop_lat"] - dest[0])**2 + ((stops_df["stop_lon"] - dest[1]) * lon_scale)**2

# 2. Isolate the top 50 closest stops to query via OSRM
candidate_origin_stops = stops_df.nsmallest(500, 'rough_dist_origin')
candidate_dest_stops = stops_df.nsmallest(500, 'rough_dist_dest')

origin_time_dict = {}
for stop in candidate_origin_stops.itertuples():
    stop_coords = (stop.stop_lat, stop.stop_lon)
    origin_time_dict[stop.stop_id] = time_journey(origin, stop_coords)

dest_time_dict = {}
for stop in candidate_dest_stops.itertuples():
    stop_coords = (stop.stop_lat, stop.stop_lon)
    dest_time_dict[stop.stop_id] = time_journey(stop_coords, dest)

## Find route

In [8]:
records = []
unique_routes = trips["route_id"].unique()

for route in unique_routes:
    stops = route_stops_dict.get(route, [])
    
    # Filter using the new time dictionaries
    valid_origin = ((stop, origin_time_dict[stop]) for stop in stops if stop in origin_time_dict)
    valid_dest = ((stop, dest_time_dict[stop]) for stop in stops if stop in dest_time_dict)
    
    try:
        o_stop, o_time = min(valid_origin, key=lambda x: x[1])
        d_stop, d_time = min(valid_dest, key=lambda x: x[1])
        
        # Ensure we don't append routes that failed OSRM routing (infinity)
        if o_time == float('inf') or d_time == float('inf'):
            continue
            
        records.append({
            "route": route, 
            "origin near stop": o_stop, 
            "origin walk time": o_time,
            "dest near stop": d_stop,
            "dest walk time": d_time,
            "lowest time sum": o_time + d_time 
        })
    except ValueError:
        continue


result_df = pd.DataFrame(records)
if not result_df.empty:
    result_df = result_df.sort_values(by="lowest time sum").reset_index(drop=True)

result_df

,route,origin near stop,origin walk time,dest near stop,dest walk time,lowest time sum
0,12777,450023165,889.3,450021125,484.6,1373.9
1,12770,450023165,889.3,450021175,650.7,1540.0
2,12779,450023165,889.3,450021175,650.7,1540.0
3,120468,450022381,1471.9,450021125,484.6,1956.5
4,22584,450023316,2019.5,450021125,484.6,2504.1
5,12767,450023316,2019.5,450021125,484.6,2504.1
6,12765,450026783,1859.0,450020539,1471.8,3330.8
7,98224,450022371,201.8,450016517,3763.2,3965.0
8,98223,450022370,210.9,450016517,3763.2,3974.1
9,24286,450023323,2046.7,450018869,2258.0,4304.7


In [9]:
for selected_route in range(len(result_df)):
    try:
        best_route_id = result_df.loc[selected_route, "route"]
        initial_stop = result_df.loc[selected_route, "origin near stop"]
        end_stop = result_df.loc[selected_route, "dest near stop"]

        route_df = stop_times_df[stop_times_df["route_id"] == best_route_id].copy()

        pivot = route_df[route_df["stop_id"].isin([initial_stop, end_stop])].pivot(
            index="trip_id", columns="stop_id", values="stop_sequence"
        )

        # Both stops must exist AND start must come before end
        valid_trips = pivot.dropna(subset=[initial_stop, end_stop])
        # valid_trips = valid_trips[valid_trips[initial_stop] < valid_trips[end_stop]]

        if valid_trips.empty:
            raise ValueError("No trips found where start stop occurs before end stop.")

        # Identify the longest trip among valid ones
        # Instead of another groupby, we can just look at the max sequence of the valid trip IDs
        longest_trip_id = route_df[route_df["trip_id"].isin(valid_trips.index)] \
                            .groupby("trip_id")["stop_sequence"].max().idxmax()

        # --- 2. Optimization: Prep Journey Data ---
        sample_journey = stop_times_df[stop_times_df["trip_id"] == longest_trip_id].copy()
        sample_journey = sample_journey.merge(stops_df[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id')

        start_seq = valid_trips.loc[longest_trip_id, initial_stop]
        end_seq = valid_trips.loc[longest_trip_id, end_stop]
        if start_seq > end_seq: start_seq, end_seq = end_seq, start_seq

        # Filter to only the segment being travelled
        mask = (sample_journey["stop_sequence"] >= start_seq) & (sample_journey["stop_sequence"] <= end_seq)
        journey_segment = sample_journey[mask].sort_values("stop_sequence")

        break
    except:
        print(f"Found {selected_route + 1} route(s) which do not have complete trips to display.")
        continue

Found 1 route(s) which do not have complete trips to display.
Found 2 route(s) which do not have complete trips to display.
Found 3 route(s) which do not have complete trips to display.
Found 4 route(s) which do not have complete trips to display.


In [10]:
bus_travel_time = sample_journey.loc[sample_journey["stop_sequence"] == end_seq, "departure_time"].item() - sample_journey.loc[sample_journey["stop_sequence"] == start_seq, "departure_time"].item()

origin_walk_seconds = result_df.loc[selected_route, "origin walk time"]
dest_walk_seconds = result_df.loc[selected_route, "dest walk time"]

# Convert OSRM seconds to Pandas Timedeltas for consistent arithmetic
origin_walk_time = pd.Timedelta(seconds=float(origin_walk_seconds))
dest_walk_time = pd.Timedelta(seconds=float(dest_walk_seconds))

# Calculate total duration
total_journey_time = origin_walk_time + bus_travel_time + dest_walk_time

# Format the output to strip microseconds if present
def format_td(td):
    return str(td).split('.')[0]

print(f"Walk to origin stop: {format_td(origin_walk_time)}")
print(f"Bus journey duration: {format_td(bus_travel_time)}")
print(f"Walk from destination stop: {format_td(dest_walk_time)}")
print(f"Total journey time: {format_td(total_journey_time)}")

Walk to origin stop: 0 days 00:33:39
Bus journey duration: 0 days 00:25:00
Walk from destination stop: 0 days 00:08:04
Total journey time: 0 days 01:06:44


In [11]:
m = folium.Map(location=[53.79, -1.75], zoom_start=12, tiles='CartoDB positron')

# Draw the actual path as a PolyLine (Better visualization than just dots)
path_coords = journey_segment[['stop_lat', 'stop_lon']].values.tolist()
folium.PolyLine(path_coords, color="blue", weight=3, opacity=0.7).add_to(m)

# Add Start/End Markers
for point, color, label in [(origin, "green", "Origin"), (dest, "red", "Destination")]:
    folium.CircleMarker(location=point, radius=5, color=color, fill=True, popup=label).add_to(m)

# Efficiently add stop markers using itertuples (faster than iterrows or loc)
for stop in journey_segment.itertuples():
    is_key_stop = stop.stop_id in [initial_stop, end_stop]
    folium.CircleMarker(
        location=[stop.stop_lat, stop.stop_lon],
        radius=4 if is_key_stop else 2,
        color="blue" if is_key_stop else "gray",
        fill=True,
        fill_opacity=0.6,
        popup=f"Seq: {stop.stop_sequence}"
    ).add_to(m)

m.save("optimized_route_map.html")